## Basic prompting

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="gpt-5-nano",
    system_prompt="You are an IT support assistant."
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": """
        Classify this ticket and explain your decision:

        Production payments are failing after this morning's deployment.
        """
    }]
})

print(result["messages"][-1].content)

Classification:
- Type: Incident (Production outage)
- Severity/Priority: Sev 1 / P1
- Impact: Production payments are failing, affecting all users; high revenue and customer experience impact; time-critical
- Category: Payments / Deployment-related incident
- Affected scope: Production environment (likely global)

Explanation:
- The issue is in production and blocks a core business function (payments), which is the highest priority for IT.
- The note “after this morning's deployment” suggests a deployment-related regression or misconfiguration is likely, so immediate triage should focus on recent changes, rollbacks, or feature flags in addition to analyzing payment gateway connectivity and error logs.

Recommended initial steps (triage):
- Confirm scope and impact with on-call/monitoring (how many failed transactions, error codes, regions).
- Review this morning’s deployment changes (what was released, config/feature flags, hotfixes).
- Check payment provider status and gateway logs f

In [3]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="gpt-5-nano",
    system_prompt="You are an IT support assistant."
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": """
        Identify the ticket category, urgency and recommended next action.
        Keep the answer under 100 words:

        Production payments are failing after this morning's deployment.
        """
    }]
})

print(result["messages"][-1].content)

Category: Incident – Production payment processing failure
Urgency: Critical (P1) – production outage affecting payments
Recommended next action: Open incident, rollback the recent deployment or apply a hotfix to restore payments, review payment gateway/API logs and transaction queues, escalate to on-call DevOps/SRE, and communicate ETA to stakeholders.


## Few-shot examples

In [7]:
few_shot_prompt = """
You classify IT support tickets.

Follow the patterns demonstrated below.

Example 1
Ticket: I forgot my password and cannot log in.
Classification: Account Access
Priority: Medium

Example 2
Ticket: The production payment API is returning HTTP 500 for every request.
Classification: Production Incident
Priority: Critical

Example 3
Ticket: Please add a dark theme to the dashboard.
Classification: Feature Request
Priority: Low


"""

agent = create_agent(
    model="gpt-5-nano",
    tools=[],
    system_prompt=few_shot_prompt
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": """
        Ticket: Customers cannot complete checkout after the latest release.
        """
    }]
})

print(result["messages"][-1].content)

Classification: Production Incident
Priority: Critical


## Structured prompts

In [8]:
structured_prompt = """
# Role
You are a senior site reliability engineer.

# Objective
Analyze production incident reports and recommend an immediate response.

# Classification Rules
- Critical: Widespread production outage, financial impact or security breach
- High: Significant degradation affecting multiple customers
- Medium: Limited impact with a workaround
- Low: Informational request or minor inconvenience

# Instructions
1. Identify the affected service.
2. Determine the likely impact.
3. Assign a priority.
4. Recommend the first three actions.
5. Mention any missing information.

# Constraints
- Do not invent facts.
- Clearly label assumptions.
- Keep the response under 200 words.

# Response Layout
Summary:
Priority:
Impact:
Immediate actions:
Missing information:
"""

agent = create_agent(
    model="gpt-5-nano",
    tools=[],
    system_prompt=structured_prompt
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": """
        Since deployment version 4.8, approximately 35% of payment requests
        have timed out. The issue started 20 minutes ago. No rollback has
        been attempted.
        """
    }]
})

print(result["messages"][-1].content)

Summary:
Deployment 4.8 caused ~35% of payment requests to timeout, started 20 minutes ago. No rollback attempted. Affected service: Payment processing API.

Priority:
High

Impact:
Significant degradation of payments: ~35% of requests failing due to timeouts. Impacts multiple customers and potential revenue loss; not a full outage but critical business function affected.

Immediate actions:
1) Roll back deployment to 4.7 (or disable 4.8) to restore normal payment flow.
2) Triage with telemetry: collect latency, error rates, timeout reasons, gateway connectivity; review recent config/DB changes and circuit breaker state.
3) Notify on-call and stakeholders; if external gateway is involved, contact provider status; consider a safe workaround (e.g., queueing or graceful degradation) if feasible.

Missing information:
Assumptions:
- Affected service is the Payment processing API; rollback to 4.7 is feasible; rollback-safe for in-flight transactions.
Needed:
- Exact error codes, latency dis

## Structured output

In [9]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class TicketAssessment(BaseModel):
    category: Literal[
        "account_access",
        "production_incident",
        "service_request",
        "feature_request"
    ] = Field(description="Primary ticket category")

    priority: Literal[
        "low",
        "medium",
        "high",
        "critical"
    ] = Field(description="Operational priority")

    summary: str = Field(
        description="One-sentence description of the problem"
    )

    affected_service: str | None = Field(
        description="Affected service, if it can be identified"
    )

    recommended_actions: list[str] = Field(
        description="Up to three immediate actions"
    )

    confidence: float = Field(
        ge=0,
        le=1,
        description="Confidence in the assessment"
    )


agent = create_agent(
    model="gpt-5-nano",
    tools=[],
    system_prompt=(
        "You are an incident-triage assistant. "
        "Use only information found in the ticket."
    ),
    response_format=TicketAssessment
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": """
        Production payment requests started returning HTTP 500 immediately
        after version 4.8 was deployed. All regions appear to be affected.
        """
    }]
})

assessment = result["structured_response"]

print(assessment)
print(assessment.priority)
print(assessment.recommended_actions)

category='production_incident' priority='critical' summary='HTTP 500 errors on production payment requests across all regions after deploying version 4.8.' affected_service='Payments service (payment processing API)' recommended_actions=['Roll back to the pre-4.8 release or deploy a hotfix to restore payment request processing.', 'Collect and analyze logs/traces across all regions to identify the root cause and failing component.', 'Check external dependencies (payment gateway, databases, and third-party services) for outages or degraded performance and apply fixes or rely on fallbacks.'] confidence=0.8
critical
['Roll back to the pre-4.8 release or deploy a hotfix to restore payment request processing.', 'Collect and analyze logs/traces across all regions to identify the root cause and failing component.', 'Check external dependencies (payment gateway, databases, and third-party services) for outages or degraded performance and apply fixes or rely on fallbacks.']
